# Step 5 — Clinical Deterioration Outcome Definition

## Objective

The purpose of this step is to define a clinically meaningful and
computationally reproducible deterioration endpoint for the
MIMIC-IV Demo cohort.

Candidate deterioration events will be investigated before the final
composite outcome is selected.

In [49]:
import pandas as pd
import numpy as np
import os

patients = pd.read_csv("../data/hosp/patients.csv")
admissions = pd.read_csv("../data/hosp/admissions.csv")
icustays = pd.read_csv("../data/icu/icustays.csv")

admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])

icustays["intime"] = pd.to_datetime(icustays["intime"])
icustays["outtime"] = pd.to_datetime(icustays["outtime"])

admissions.columns.tolist()
admissions["hospital_expire_flag"].value_counts(dropna=False)

n_admissions = len(admissions)

n_deaths = (
    admissions["hospital_expire_flag"] == 1
).sum()

mortality_rate = n_deaths / n_admissions * 100

print("Hospital admissions:", n_admissions)
print("In-hospital deaths:", n_deaths)
print(f"In-hospital mortality: {mortality_rate:.2f}%")

icu_outcomes = icustays.merge(
    admissions[
        [
            "subject_id",
            "hadm_id",
            "hospital_expire_flag",
            "deathtime"
        ]
    ],
    on=["subject_id", "hadm_id"],
    how="left"
)

icu_outcomes[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime",
        "hospital_expire_flag",
        "deathtime"
    ]
].head(10)

icu_outcomes["hospital_expire_flag"].value_counts(dropna=False)

icu_outcomes["deathtime"] = pd.to_datetime(
    icu_outcomes["deathtime"]
)

icu_deaths = icu_outcomes[
    icu_outcomes["hospital_expire_flag"] == 1
].copy()

icu_deaths[
    [
        "subject_id",
        "stay_id",
        "intime",
        "outtime",
        "deathtime"
    ]
].head(20)

icu_deaths["hours_from_icu_admission_to_death"] = (
    icu_deaths["deathtime"] - icu_deaths["intime"]
).dt.total_seconds() / 3600

icu_deaths[
    [
        "subject_id",
        "stay_id",
        "hours_from_icu_admission_to_death"
    ]
].head(20)

icu_deaths["hours_from_icu_admission_to_death"].describe()

icu_files = os.listdir("../data/icu/")
sorted(icu_files)

procedureevents = pd.read_csv(
    "../data/icu/procedureevents.csv",
    low_memory=False
)

d_items = pd.read_csv(
    "../data/icu/d_items.csv",
    low_memory=False
)

procedureevents.shape

procedureevents.columns.tolist()
procedureevents.head()

d_items[
    d_items["label"].str.contains(
        "vent",
        case=False,
        na=False
    )
][
    ["itemid", "label", "category", "unitname"]
].head(50)

d_items[
    d_items["label"].str.contains(
        "intubat",
        case=False,
        na=False
    )
][
    ["itemid", "label", "category", "unitname"]
].head(50)

inputevents = pd.read_csv(
    "../data/icu/inputevents.csv",
    low_memory=False
)

inputevents.shape

vasopressor_candidates = d_items[
    d_items["label"].str.contains(
        "norepinephrine|epinephrine|vasopressin",
        case=False,
        na=False,
        regex=True
    )
][
    ["itemid", "label", "category", "unitname"]
].copy()

vasopressor_candidates

vasopressor_usage = (
    inputevents[
        inputevents["itemid"].isin(
            vasopressor_candidates["itemid"]
        )
    ]
    .groupby("itemid")
    .agg(
        measurements=("itemid", "size"),
        patients=("subject_id", "nunique"),
        icu_stays=("stay_id", "nunique")
    )
    .reset_index()
)

vasopressor_usage

vasopressor_summary = vasopressor_candidates.merge(
    vasopressor_usage,
    on="itemid",
    how="left"
)

vasopressor_summary[
    ["itemid", "label", "category", "unitname",
     "measurements", "patients", "icu_stays"]
]
VASOPRESSOR_ITEMIDS = {
    "Norepinephrine": 221906,
    "Vasopressin": 222315,
    "Epinephrine": 221289
}

VASOPRESSOR_ITEMIDS

vasopressor_events = inputevents[
    inputevents["itemid"].isin(
        VASOPRESSOR_ITEMIDS.values()
    )
].copy()

vasopressor_events.shape

vasopressor_events[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "itemid",
        "starttime",
        "endtime",
        "amount",
        "amountuom",
        "rate",
        "rateuom"
    ]
].head(20)

itemid_to_drug = {
    221906: "Norepinephrine",
    222315: "Vasopressin",
    221289: "Epinephrine"
}

vasopressor_events["vasopressor"] = (
    vasopressor_events["itemid"]
    .map(itemid_to_drug)
)
vasopressor_events[
    [
        "subject_id",
        "stay_id",
        "starttime",
        "vasopressor"
    ]
].head(20)

vasopressor_events.groupby(
    "vasopressor"
).agg(
    measurements=("itemid", "size"),
    patients=("subject_id", "nunique"),
    icu_stays=("stay_id", "nunique")
)

Hospital admissions: 275
In-hospital deaths: 15
In-hospital mortality: 5.45%


,measurements,patients,icu_stays
vasopressor,,,
Epinephrine,36,4,4
Norepinephrine,947,26,33
Vasopressin,55,8,9


## 5.7 Construction of the Clinical Deterioration Outcome

Clinical deterioration was operationalized as the first qualifying
occurrence of new invasive mechanical ventilation, new vasopressor
initiation, or death during an ICU stay.

Event timestamps were retained to support subsequent construction
of temporal prediction windows and to minimize temporal leakage.

In [50]:
outcomes = icustays[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime"
    ]
].copy()

outcomes["intime"] = pd.to_datetime(outcomes["intime"])
outcomes["outtime"] = pd.to_datetime(outcomes["outtime"])

outcomes.head()

print("Rows:", len(outcomes))
print("Unique ICU stays:", outcomes["stay_id"].nunique())

vasopressor_events["starttime"] = pd.to_datetime(
    vasopressor_events["starttime"]
)

first_vasopressor = (
    vasopressor_events
    .groupby("stay_id")["starttime"]
    .min()
    .reset_index()
    .rename(
        columns={
            "starttime": "vasopressor_time"
        }
    )
)

first_vasopressor.head()

outcomes = outcomes.merge(
    first_vasopressor,
    on="stay_id",
    how="left"
)

# Keep only vasopressor events occurring during the ICU stay
valid_vasopressor = (
    outcomes["vasopressor_time"].notna()
    & outcomes["vasopressor_time"].between(
        outcomes["intime"],
        outcomes["outtime"]
    )
)

outcomes["vasopressor_time"] = outcomes[
    "vasopressor_time"
].where(valid_vasopressor)

outcomes["vasopressor_hours_from_icu"] = (
    outcomes["vasopressor_time"] - outcomes["intime"]
).dt.total_seconds() / 3600

outcomes.loc[
    outcomes["vasopressor_time"].notna(),
    [
        "stay_id",
        "intime",
        "outtime",
        "vasopressor_time",
        "vasopressor_hours_from_icu"
    ]
].head(20)

death_info = admissions[
    [
        "subject_id",
        "hadm_id",
        "deathtime",
        "hospital_expire_flag"
    ]
].copy()

death_info["deathtime"] = pd.to_datetime(
    death_info["deathtime"]
)

outcomes = outcomes.merge(
    death_info,
    on=["subject_id", "hadm_id"],
    how="left"
)

outcomes["icu_death"] = (
    outcomes["deathtime"].notna()
    & (outcomes["deathtime"] >= outcomes["intime"])
    & (outcomes["deathtime"] <= outcomes["outtime"])
)

outcomes["icu_death"].value_counts()

outcomes["death_event_time"] = outcomes[
    "deathtime"
].where(
    outcomes["icu_death"]
)

ventilation_events = procedureevents.loc[
    procedureevents["ordercategoryname"]
    .astype("string")
    .str.casefold()
    .eq("ventilation"),
    ["stay_id", "starttime"]
].copy()

ventilation_events["starttime"] = pd.to_datetime(
    ventilation_events["starttime"],
    errors="coerce"
)

ventilation_events = ventilation_events.dropna(
    subset=["stay_id", "starttime"]
)

# Keep only ventilation events occurring during the corresponding ICU stay
ventilation_events = ventilation_events.merge(
    outcomes[["stay_id", "intime", "outtime"]],
    on="stay_id",
    how="inner"
)

ventilation_events = ventilation_events.loc[
    ventilation_events["starttime"].between(
        ventilation_events["intime"],
        ventilation_events["outtime"]
    ),
    ["stay_id", "starttime"]
]

first_ventilation = (
    ventilation_events
    .groupby("stay_id")["starttime"]
    .min()
    .reset_index()
    .rename(
        columns={
            "starttime": "ventilation_time"
        }
    )
)

outcomes = outcomes.merge(
    first_ventilation,
    on="stay_id",
    how="left"
)

outcomes["ventilation_hours_from_icu"] = (
    outcomes["ventilation_time"] - outcomes["intime"]
).dt.total_seconds() / 3600

outcomes[
    [
        "subject_id",
        "stay_id",
        "intime",
        "vasopressor_time",
        "ventilation_time",
        "death_event_time"
    ]
].head(20)

event_columns = [
    "vasopressor_time",
    "ventilation_time",
    "death_event_time"
]

outcomes["event_time"] = outcomes[
    event_columns
].min(axis=1)

outcomes["deterioration"] = (
    outcomes["event_time"].notna()
).astype(int)

outcomes["deterioration"].value_counts()

def determine_event_type(row):

    if pd.isna(row["event_time"]):
        return "None"

    if row["event_time"] == row["ventilation_time"]:
        return "Mechanical ventilation"

    if row["event_time"] == row["vasopressor_time"]:
        return "Vasopressor"

    if row["event_time"] == row["death_event_time"]:
        return "Death"

    return "Unknown"

outcomes["event_type"] = outcomes.apply(
    determine_event_type,
    axis=1
)

outcomes["event_type"].value_counts()

outcomes["hours_to_deterioration"] = (
    outcomes["event_time"] - outcomes["intime"]
).dt.total_seconds() / 3600

outcomes.loc[
    outcomes["deterioration"] == 1,
    "hours_to_deterioration"
].describe()

outcomes[
    outcomes["hours_to_deterioration"] < 0
][
    [
        "subject_id",
        "stay_id",
        "intime",
        "event_time",
        "event_type",
        "hours_to_deterioration"
    ]
]

outcomes[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime",
        "event_time",
        "event_type",
        "deterioration",
        "hours_to_deterioration"
    ]
].head(20)

outcomes.to_csv(
    "../results/deterioration_outcomes.csv",
    index=False
)
outcomes["deterioration"].value_counts()

outcomes.loc[
    outcomes["deterioration"] == 1,
    "hours_to_deterioration"
].describe()

vasopressor_events["starttime"] = pd.to_datetime(
    vasopressor_events["starttime"]
)

first_vasopressor = (
    vasopressor_events
    .groupby("stay_id")["starttime"]
    .min()
    .reset_index()
    .rename(columns={"starttime": "vasopressor_time"})
)

first_vasopressor.head()

outcomes = outcomes.merge(
    first_vasopressor,
    on="stay_id",
    how="left"
)
print("Rows after vasopressor merge:", len(outcomes))
print("Unique ICU stays:", outcomes["stay_id"].nunique())
print("Duplicate stay IDs:", outcomes["stay_id"].duplicated().sum())

# Resolve duplicate columns created by rerunning the merge.
if "vasopressor_time" not in outcomes.columns:
    if "vasopressor_time_x" in outcomes.columns:
        outcomes["vasopressor_time"] = outcomes["vasopressor_time_x"]
    elif "vasopressor_time_y" in outcomes.columns:
        outcomes["vasopressor_time"] = outcomes["vasopressor_time_y"]
    else:
        outcomes["vasopressor_time"] = pd.NaT

outcomes["vasopressor_time"] = pd.to_datetime(
    outcomes["vasopressor_time"],
    errors="coerce"
)

outcomes["vasopressor_hours_from_icu"] = (
    outcomes["vasopressor_time"] - outcomes["intime"]
).dt.total_seconds() / 3600

outcomes.loc[
    outcomes["vasopressor_time"].notna(),
    "vasopressor_hours_from_icu"
].describe()

outcomes["deterioration"].value_counts()
outcomes["event_type"].value_counts()
outcomes[
    outcomes["deterioration"] == 1
].groupby(
    "event_type"
)["hours_to_deterioration"].describe()

for hours in [1, 2, 4, 6]:
    n = (
        (outcomes["deterioration"] == 1) &
        (outcomes["hours_to_deterioration"] <= hours)
    ).sum()

    print(f"Events within {hours} hour(s): {n}")

    outcomes["event_type"].value_counts()
    outcomes[
    outcomes["deterioration"] == 1
].groupby(
    "event_type"
)["hours_to_deterioration"].describe()

    early_event_summary = (
    outcomes[
        outcomes["deterioration"] == 1
    ]
    .assign(
        within_6h=lambda x:
            x["hours_to_deterioration"] <= 6
    )
    .groupby("event_type")
    .agg(
        total_events=("stay_id", "count"),
        events_within_6h=("within_6h", "sum"),
        median_hours=("hours_to_deterioration", "median")
    )
)

early_event_summary

outcomes["landmark_time"] = (
    outcomes["intime"] + pd.Timedelta(hours=6)
)

outcomes["prediction_end"] = (
    outcomes["landmark_time"] + pd.Timedelta(hours=6)
)

outcomes["reaches_landmark"] = (
    outcomes["outtime"] >= outcomes["landmark_time"]
)

outcomes["event_before_landmark"] = (
    outcomes["event_time"].notna()
    & (outcomes["event_time"] <= outcomes["landmark_time"])
)

outcomes["eligible_at_landmark"] = (
    outcomes["reaches_landmark"]
    & ~outcomes["event_before_landmark"]
)

outcomes["eligible_at_landmark"].value_counts()

outcomes["deterioration_6_12h"] = (
    outcomes["eligible_at_landmark"]
    & outcomes["event_time"].notna()
    & (outcomes["event_time"] > outcomes["landmark_time"])
    & (outcomes["event_time"] <= outcomes["prediction_end"])
).astype(int)

eligible_outcomes = outcomes[
    outcomes["eligible_at_landmark"]
].copy()

eligible_outcomes = outcomes[
    outcomes["eligible_at_landmark"]
].copy()

eligible_outcomes.loc[
    eligible_outcomes["deterioration_6_12h"] == 1,
    "event_type"
].value_counts()

print("Total ICU stays:", len(outcomes))

print(
    "Reached 6-hour landmark:",
    outcomes["reaches_landmark"].sum()
)

print(
    "Event before/by landmark:",
    outcomes["event_before_landmark"].sum()
)

print(
    "Eligible at landmark:",
    outcomes["eligible_at_landmark"].sum()
)

print(
    "Deterioration during 6–12 h:",
    eligible_outcomes["deterioration_6_12h"].sum()
)

print(
    "No deterioration during 6–12 h:",
    (eligible_outcomes["deterioration_6_12h"] == 0).sum()
)

outcomes.to_csv(
    "../results/deterioration_outcomes.csv",
    index=False
)

eligible_outcomes.to_csv(
    "../results/landmark_6h_outcomes.csv",
    index=False
)

Rows: 140
Unique ICU stays: 140
Rows after vasopressor merge: 140
Unique ICU stays: 140
Duplicate stay IDs: 0
Events within 1 hour(s): 31
Events within 2 hour(s): 40
Events within 4 hour(s): 51
Events within 6 hour(s): 57
Total ICU stays: 140
Reached 6-hour landmark: 137
Event before/by landmark: 57
Eligible at landmark: 80
Deterioration during 6–12 h: 2
No deterioration during 6–12 h: 78


## Step 5 Conclusion

A composite clinical deterioration outcome was defined as the first
qualifying occurrence of invasive mechanical ventilation, vasopressor
initiation, or death during an ICU stay.

Among 140 ICU stays, 70 had at least one candidate deterioration event.
The first qualifying event was mechanical ventilation in 54 stays,
vasopressor initiation in 15 stays, and death in 1 stay.

Event timing demonstrated that deterioration frequently occurred soon
after ICU admission. Of the 70 candidate events, 57 occurred within the
first 6 hours.

A preliminary 6-hour landmark design was therefore evaluated. After
requiring patients to remain event-free and under observation until
6 hours after ICU admission, 80 ICU stays remained eligible. Only 2 of
these 80 stays experienced deterioration during the subsequent 6-hour
prediction window (hours 6–12), while 78 did not.

Therefore, a fixed 6-hour observation followed by a 6-hour prediction
window was considered unsuitable for the primary proof-of-concept
machine-learning analysis because of the very small number of positive
outcomes. Alternative temporal designs will be evaluated before model
development.

The original event timestamps and event types were retained rather than
altering or removing early events.